In [1]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
DATASET_PATH = "data/tsuivan_dataset.json"
OUTPUT_DIR = "./qwen-tsuivan-lora"

/home/toru2/Amara/Deep_learning/midterm2/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]/home/toru2/Amara/Deep_learning/midterm2/venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 434/434 [00:00<00:00, 646.74it/s]


In [3]:
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

def format_chat(example):
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

dataset = Dataset.from_list(raw_data).map(format_chat)
print(f"Нийт жишээ: {len(dataset)}")
print("--- Жишээ ---")
print(dataset[0]["text"][:500])

Map: 100%|██████████| 82/82 [00:00<00:00, 7970.27 examples/s]

Нийт жишээ: 82
--- Жишээ ---
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Цуйван хэрхэн хийх вэ?<|im_end|>
<|im_start|>assistant
Орц:
- Гурил: 500г
- Үхрийн мах: 300г
- Төмс: 2 ширхэг
- Лууван: 1 ширхэг
- Байцаа: 200г
- Сонгино: 1 ширхэг
- Сармис: 3 хумс
- Ургамлын тос: 3 халбага
- Давс, хар перц: амтлах
- Шар буурцагны соус: 2 халбага

Алхам:
1. Гурилд давс, ус нэмж хатуувтар зуурна. 20 минут амраана.
2. Зуурмагаа нимгэн дэлгэж, уураар 15 минут жигнэнэ.



In [4]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [5]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=20,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,
    weight_decay=0.01,
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)

Tokenizing train dataset: 100%|██████████| 82/82 [00:00<00:00, 2111.73 examples/s]


In [6]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/home/toru2/Amara/Deep_learning/midterm2/venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,3.112300
10,2.627733
15,2.141740
20,1.809783
25,1.495723
30,1.363651
35,1.213735
40,1.071611
45,1.061130
50,0.918843


/home/toru2/Amara/Deep_learning/midterm2/venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/toru2/Amara/Deep_learning/midterm2/venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences bet

TrainOutput(global_step=220, training_loss=0.5419669429009611, metrics={'train_runtime': 345.5783, 'train_samples_per_second': 4.746, 'train_steps_per_second': 0.637, 'total_flos': 1.243893309382656e+16, 'train_loss': 0.5419669429009611})

In [7]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Загвар хадгалагдлаа: {OUTPUT_DIR}")

Загвар хадгалагдлаа: ./qwen-tsuivan-lora


In [8]:
model.config.use_cache = True
model.eval()

def ask(question, max_new_tokens=512):
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    answer = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return answer

print(ask("Цуйван хэрхэн хийх вэ?"))

/home/toru2/Amara/Deep_learning/midterm2/venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Орц:
- Гурил: 500г
- Үхрийн мах: 300г
- Төмс: 2 ширхэг
- Лууван: 1 ширхэг
- Байцаа: 200г
- Сонгино: 1 ширхэг
- Сармис: 3 хумс
- Ургамлын тос: 3 халбага
- Давс, хар перц: амтлах
- Шар буурцагны соус: 2 халбага

Алхам:
1. Гурилд давс, ус нэмж хатуувтар зуурна. 20 минут амраана.
2. Зуурмагаа нимгэн дэлгэж, уураар 15 минут жигнэнэ.
3. Жигнэсэн гурилаа хөргөөд нарийн зүснэ.
4. Махаа жижиглэн хэрчиж, халуун тосонд хайрна.
5. Сонгино, сармисаа нэмж хуурна.
6. Лууван, төмсөө нэмж 5 минут хуурна.
7. Байцаагаа нэмж шар буурцагны соус, давс, перцээр амтална.
8. Жигнэсэн гурилаа нэмж сайтар хольж 5 минут хуурна.

Зөвлөгөө:
- Гурилаа 2–3 мм зузаантай дэлгэвэл зохистой.
- Мах сайн хуурсны дараа хүнсний ногоогоо нэмнэ.
- Шар буурцагны соусыг хэтрүүлэхгүй.


In [ ]:
test_questions = [
    "Үхрийн махтай цуйван хийх жор хэлнэ үү.",
    "Махгүй цуйван хэрхэн хийх вэ?",
    "Цуйванд ямар хүнсний ногоо ордог вэ?",
    "Цуйван хурдан хийх арга хэлнэ үү.",
]

for q in test_questions:
    print(f"\n{'='*60}\nАсуулт: {q}\n{'='*60}")
    print(ask(q))


Асуулт: Үхрийн махтай цуйван хийх жор хэлнэ үү.
Орц:
- Гурил: 500г
- Үхрийн мах (нуруу, зүрх): 350г
- Төмс: 2 ширхэг
- Лууван: 1 ширхэг
- Сонгино: 1 ширхэг
- Байцаа: 150г
- Сармис: 4 хумс
- Давс, перц: амтлах
- Ургамлын тос: 3 халбага

Алхам:
1. Үхрийн махаа нимгэн зүсэж давстай усанд 10 минут дэвтээнэ.
2. Гурилаа зууран амраана.
3. Гурилаа нимгэн дэлгэж уураар жигнэнэ.
4. Жигнэсэн гурилаа нарийн зүснэ.
5. Махаа халуун тосонд хайрч сонгино, сармисаа нэмнэ.
6. Төмс, лууванаа нэмж 5 минут хуурна.
7. Байцаагаа нэмж, давс, перцээр амтална.
8. Гурилаа нэмж хольж 5 минут болгоно.

Зөвлөгөө:
- Үхрийн мах богино хугацаанд хайрах нь зөөлөн болгоно.
- Байцаагаа хэт хуурахгүй, шинэхэн байлгана.
- Сүүлд жаахан цагаан мангирын гиш нэмэх нь үнэр сайжруулна.

Асуулт: Махгүй цуйван хэрхэн хийх вэ?
Орц:
- Гурил: 500г
- Мөөг: 200г
- Лууван: 1 ширхэг
- Төмс: 2 ширхэг
- Байцаа: 200г
- Чинжүү: 1 ширхэг
- Сонгино: 1 ширхэг
- Сармис: 3 хумс
- Ургамлын тос: 3 халбага
- Давс, перц, шар буурцагны соус

Алхам:


: 